[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# BSON Types &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, which starts MongoDB and seeds it. Run it first. Each
task opens its own client and closes it, so they can be run in any order.


In [1]:
import datetime as dt
import decimal
import json as stdlib_json
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import bson
import pymongo
from bson import Decimal128, ObjectId

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** A value BSON has no room for.


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

try:
    shop.answers.insert_one({"tags": {"a", "b"}})
except bson.errors.InvalidDocument as error:
    print(type(error).__module__ + "." + type(error).__name__)
    print(" ", str(error).split(", of type")[0])

client.close()


bson.errors.InvalidDocument
  Invalid document: cannot encode object: {'b', 'a'}


The message names the value, which is more help than it sounds: in a document of thirty fields it
tells you which one to look at.


**2.** A tuple, going in and coming out.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

shop.answers.replace_one({"_id": "t"}, {"_id": "t", "pair": (1, 2)}, upsert=True)
back = shop.answers.find_one({"_id": "t"})["pair"]

print("went in as a tuple, came back as a", type(back).__name__)
print("value:", back, "| equal to the tuple:", back == (1, 2))
client.close()


went in as a tuple, came back as a list
value: [1, 2] | equal to the tuple: False


No error and no warning. If anything downstream checks `isinstance(x, tuple)` it will now be wrong,
and the only sign is that the check fails.


**3.** How much of a timestamp survives.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

exact = dt.datetime(2026, 7, 8, 9, 10, 11, 123456, tzinfo=dt.timezone.utc)
shop.answers.replace_one({"_id": "w"}, {"_id": "w", "at": exact}, upsert=True)
stored = shop.answers.find_one({"_id": "w"})["at"]

print("wrote microseconds:", exact.microsecond)
print("read microseconds: ", stored.microsecond)
print("lost:", exact.microsecond - stored.microsecond)
client.close()


wrote microseconds: 123456
read microseconds:  123000
lost: 456


BSON keeps milliseconds, so the last three digits go. Round on your side first if you need the value
you hold to equal the value stored.


**4.** The same date, read two ways.


In [5]:
aware = pymongo.MongoClient(URI, tz_aware=True)
naive = pymongo.MongoClient(URI)

with_tz = aware.get_default_database().answers.find_one({"_id": "w"})["at"]
without = naive.get_default_database().answers.find_one({"_id": "w"})["at"]

print("tz_aware=True:", with_tz.isoformat())
print("the default:  ", without.isoformat())
print("same instant, one of them cannot say so:", with_tz.replace(tzinfo=None) == without)

aware.close()
naive.close()


tz_aware=True: 2026-07-08T09:10:11.123000+00:00
the default:   2026-07-08T09:10:11.123000
same instant, one of them cannot say so: True


The naive one is UTC with the label removed. That is why it compares fine with other naive
datetimes and raises against anything aware, usually a long way from here.


**5.** Money that adds up.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

shop.answers.replace_one({"_id": "m"},
                         {"_id": "m", "f": 0.1, "d": Decimal128("0.1")}, upsert=True)
money = shop.answers.find_one({"_id": "m"})

print("float,      three times:", money["f"] * 3)
print("Decimal128, three times:", money["d"].to_decimal() * 3)
print("only one of those is 0.3:", money["d"].to_decimal() * 3 == decimal.Decimal("0.3"))
client.close()


float,      three times: 0.30000000000000004
Decimal128, three times: 0.3
only one of those is 0.3: True


`Decimal128` stores the number exactly. The arithmetic happens in Python's `decimal`, which is why
`to_decimal()` is there, and the result goes back into a `Decimal128` to be stored.


**6.** A document with an ObjectId, sent as JSON.


In [7]:
document = {"_id": ObjectId(), "kind": "laptop"}

try:
    stdlib_json.dumps(document)
except TypeError as error:
    print("straight to json.dumps ->", error)

at_the_edge = stdlib_json.dumps({**document, "_id": str(document["_id"])})
print("converted at the edge:  ", sorted(stdlib_json.loads(at_the_edge)))

as_bson = bson.json_util.dumps(document)
print("through json_util:      ", sorted(stdlib_json.loads(as_bson)["_id"]))
print("and it comes back an", type(bson.json_util.loads(as_bson)["_id"]).__name__)


straight to json.dumps -> Object of type ObjectId is not JSON serializable
converted at the edge:   ['_id', 'kind']
through json_util:       ['$oid']
and it comes back an ObjectId


Two different answers to two different questions. A browser wants the string. A file that something
will load back as BSON wants the `{"$oid": ...}`, which is Extended JSON, and no public API should
be sending it.


---

&#8592; **Back to:** [BSON Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/03-bson-types.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
